In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
print(sorted(p.name for p in Path("/kaggle/working").iterdir()))

In [ ]:
import re, time, shutil, tarfile, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
t00 = time.time()

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
TAR, TSV = WORK / "drkg.tar.gz", WORK / "drkg.tsv"
URL = "https://dgl-data.s3-us-west-2.amazonaws.com/dataset/DRKG/drkg.tar.gz"

# ---- 1) download + extract ONLY drkg.tsv ----
if not TSV.exists():
    if not TAR.exists():
        with urllib.request.urlopen(URL, timeout=300) as r, open(TAR, "wb") as f:
            shutil.copyfileobj(r, f, 1 << 20)
    with tarfile.open(TAR) as t:
        m = next(x for x in t.getmembers() if x.name.split("/")[-1] == "drkg.tsv")
        m.name = "drkg.tsv"; t.extract(m, WORK, filter="data")
    TAR.unlink()                                               # free ~217 MB of disk
print(f"[1/4] data ready ({time.time()-t00:.0f}s)")

# ---- 2) load ----
df = pd.read_csv(TSV, sep="\t", header=None, names=["head", "rel", "tail"])
df["h"] = df["head"].str.split("::").str[0]; df["t"] = df["tail"].str.split("::").str[0]
print(f"[2/4] drkg loaded: {df.shape} ({time.time()-t00:.0f}s)")

# ---- 3) labels (DrugBank 'treats') + graph without ANY compound-disease edge ----
is_cd = ((df.h == "Compound") & (df.t == "Disease")) | ((df.h == "Disease") & (df.t == "Compound"))
cd = df[is_cd].copy()
cd["compound"] = np.where(cd.h == "Compound", cd["head"], cd["tail"]); cd["disease"] = np.where(cd.h == "Disease", cd["head"], cd["tail"])
is_gd = ((df.h == "Gene") & (df.t == "Disease")) | ((df.h == "Disease") & (df.t == "Gene"))
is_cg = ((df.h == "Compound") & (df.t == "Gene")) | ((df.h == "Gene") & (df.t == "Compound"))
dis_with_genes = set(np.where(df.h[is_gd] == "Disease", df["head"][is_gd], df["tail"][is_gd]))
comp_with_genes = set(np.where(df.h[is_cg] == "Compound", df["head"][is_cg], df["tail"][is_cg]))
lab = cd[cd.rel == "DRUGBANK::treats::Compound:Disease"][["compound", "disease"]].drop_duplicates()
lab = lab[lab.compound.isin(comp_with_genes) & lab.disease.isin(dis_with_genes)].reset_index(drop=True)
print(f"[3/4] usable labels: {len(lab)} pairs | {lab.compound.nunique()} compounds | {lab.disease.nunique()} diseases   (expected 3674 | 1324 | 762)")
graph = df.loc[~is_cd, ["head", "rel", "tail"]].reset_index(drop=True)
assert not graph["head"].str.startswith("Disease::").mul(graph["tail"].str.startswith("Compound::")).any()
assert not graph["head"].str.startswith("Compound::").mul(graph["tail"].str.startswith("Disease::")).any()
del df, cd

# ---- 4) index into matrices + cold-drug 5-fold split + save ----
ht = graph["head"].str.split("::").str[0].values; tt = graph["tail"].str.split("::").str[0].values
H, T = graph["head"].values, graph["tail"].values
def pairs(a, b):
    m1 = (ht == a) & (tt == b)
    if a == b: return H[m1], T[m1]
    m2 = (ht == b) & (tt == a)
    return np.concatenate([H[m1], T[m2]]), np.concatenate([T[m1], H[m2]])
cg_c, cg_g = pairs("Compound", "Gene"); gg_a, gg_b = pairs("Gene", "Gene"); gd_g, gd_d = pairs("Gene", "Disease")
comps = sorted(set(lab.compound) | {c for c in set(cg_c) if c.startswith("Compound::DB")})
genes = sorted(set(cg_g) | set(gg_a) | set(gg_b) | set(gd_g)); diseases = sorted(lab.disease.unique())
cmap, gmap, dmap = ({e: i for i, e in enumerate(L)} for L in (comps, genes, diseases))
nc, ng, nd = len(comps), len(genes), len(diseases)
def to_csr(a, b, amap, bmap, shape, sym=False):
    r, c = pd.Series(a).map(amap), pd.Series(b).map(bmap); ok = (r.notna() & c.notna()).values
    M = sp.csr_matrix((np.ones(ok.sum(), dtype=np.float32), (r[ok].astype(int).values, c[ok].astype(int).values)), shape=shape)
    if sym: M = M + M.T
    M.data[:] = 1.0
    return M.tocsr()
A_cg = to_csr(cg_c, cg_g, cmap, gmap, (nc, ng)); A_gg = to_csr(gg_a, gg_b, gmap, gmap, (ng, ng), sym=True); A_gd = to_csr(gd_g, gd_d, gmap, dmap, (ng, nd))
Y = to_csr(lab.compound.values, lab.disease.values, cmap, dmap, (nc, nd)).toarray()
assert int(Y.sum()) == len(lab)
has = np.where(Y.sum(1) > 0)[0]
folds = np.array_split(rng.permutation(has), 5)
for nm_, M in dict(A_cg=A_cg, A_gg=A_gg, A_gd=A_gd).items(): sp.save_npz(WORK / f"{nm_}.npz", M)
np.save(WORK / "Y.npy", Y); np.savez(WORK / "folds.npz", *folds)
pd.to_pickle(dict(comps=comps, genes=genes, diseases=diseases), WORK / "index.pkl"); lab.to_pickle(WORK / "labels.pkl")
del graph, H, T, ht, tt
print(f"[4/4] {nc} drugs | {ng} genes | {nd} diseases | A_cg {A_cg.nnz:,} A_gg {A_gg.nnz:,} A_gd {A_gd.nnz:,} | folds {[len(f) for f in folds]}")
print(f"saved to {WORK}: {sorted(p.name for p in WORK.iterdir())}  | total {time.time()-t00:.0f}s")

In [ ]:
import time, numpy as np, pandas as pd, scipy.sparse as sp
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier

WORK = Path("/kaggle/working")
need = ["A_cg.npz", "A_gg.npz", "A_gd.npz", "Y.npy", "folds.npz", "index.pkl"]
missing = [f for f in need if not (WORK / f).exists()]
assert not missing, f"Saved files missing: {missing}. The earlier cells must be re-run first (see message)."

A_cg, A_gg, A_gd = (sp.load_npz(WORK / f"{n}.npz").tocsr() for n in ["A_cg", "A_gg", "A_gd"])
Y = np.load(WORK / "Y.npy"); fz = np.load(WORK / "folds.npz"); folds = [fz[f"arr_{i}"] for i in range(len(fz.files))]
idx = pd.read_pickle(WORK / "index.pkl"); comps, genes, diseases = idx["comps"], idx["genes"], idx["diseases"]
nc, ng, nd = len(comps), len(genes), len(diseases)
has = np.where(Y.sum(1) > 0)[0]
rng = np.random.default_rng(42)
print(f"restored: {nc} drugs, {ng} genes, {nd} diseases, {int(Y.sum())} labels, {len(folds)} folds")

def Y_train_for(test_drugs):
    Yt = Y.copy(); Yt[test_drugs] = 0
    return Yt

def ranks_for(S, drugs, Yfull=None):
    Yfull = Y if Yfull is None else Yfull
    R, A = [], []
    for d in drugs:
        pos = np.where(Yfull[d] > 0)[0]
        if len(pos) == 0: continue
        neg = np.sort(np.delete(S[d], pos))
        lo = np.searchsorted(neg, S[d][pos], "left"); hi = np.searchsorted(neg, S[d][pos], "right")
        g, e = len(neg) - hi, hi - lo
        R.extend(1 + g + 0.5 * e); A.extend(1 - (g + 0.5 * e) / len(neg))
    return R, A

def summarize(name, R, A):
    R = np.array(R)
    return {"model": name, "pairs": len(R), "AUROC": np.mean(A), "MRR": np.mean(1 / R),
            "Hits@1": np.mean(R <= 1), "Hits@10": np.mean(R <= 10), "Hits@50": np.mean(R <= 50), "median_rank": np.median(R)}

def run_C(Acg, Agg, Agd, label, seed=42):
    """Strong baseline (similar-drug vote + path features + boosting)."""
    rg = np.random.default_rng(seed)
    S2_ = (Acg @ Agd).toarray().astype(np.float32)
    S3_ = ((Acg @ Agg) @ Agd).toarray().astype(np.float32)
    dgc = np.asarray(Acg.sum(1)).ravel().astype(np.float32); dgd = np.asarray(Agd.sum(0)).ravel().astype(np.float32)
    nrm_ = np.sqrt(np.asarray(Acg.multiply(Acg).sum(1)).ravel()); nrm_[nrm_ == 0] = 1
    Vn_ = sp.diags(1 / nrm_).dot(Acg).tocsr()
    SIM_ = (Vn_ @ Vn_.T).toarray().astype(np.float32); np.fill_diagonal(SIM_, 0); SS = SIM_.sum(1) + 1e-6
    def feats(Yt):
        K = SIM_ @ Yt; Kn = K / SS[:, None]; pop = np.broadcast_to(Yt.sum(0), (nc, nd))
        return np.stack([np.log1p(S2_), np.log1p(S3_), S2_ / np.sqrt(1 + dgd)[None, :], K, Kn, pop,
                         np.broadcast_to(np.log1p(dgc)[:, None], (nc, nd)), np.broadcast_to(np.log1p(dgd)[None, :], (nc, nd))], -1).astype(np.float32)
    R, A, scores = [], [], {}
    for k, f in enumerate(folds):
        Yt = Y_train_for(f); F = feats(Yt)
        tr_lab = np.setdiff1d(has, f)
        pr, pcol = np.nonzero(Yt[tr_lab]); pr = tr_lab[pr]
        neg_r = rg.choice(tr_lab, len(pr) * 5); neg_c = rg.integers(0, nd, len(pr) * 5); keep = Yt[neg_r, neg_c] == 0
        X = np.concatenate([F[pr, pcol], F[neg_r[keep], neg_c[keep]]]); y = np.concatenate([np.ones(len(pr)), np.zeros(keep.sum())])
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_depth=5, early_stopping=True,
                                           validation_fraction=0.15, random_state=seed).fit(X, y)
        Sf = np.zeros((nc, nd), np.float32); Sf[f] = m.predict_proba(F[f].reshape(-1, F.shape[-1]))[:, 1].reshape(len(f), nd)
        scores[k] = Sf; r, a = ranks_for(Sf, f); R += r; A += a
        del F, X
    return summarize(label, R, A), scores

t0 = time.time()
out_all, sc_all = run_C(A_cg, A_gg, A_gd, "C. all genes (baseline to beat)")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in out_all.items()}, f"| {time.time()-t0:.0f}s")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as Fn
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

def norm_op(M):
    """Row-normalised sparse operator (mean over neighbours) as a torch sparse tensor."""
    M = sp.csr_matrix(M, dtype=np.float32); deg = np.asarray(M.sum(1)).ravel(); deg[deg == 0] = 1
    X = (sp.diags(1 / deg) @ M).tocoo()
    return torch.sparse_coo_tensor(np.vstack([X.row, X.col]), X.data, X.shape).coalesce().to(dev)

# Fixed graph operators (they contain NO labels; the drug->disease labels are only ever used as training targets)
OPS = dict(gg=norm_op(A_gg), gd=norm_op(A_gd), dg=norm_op(A_gd.T), cg=norm_op(A_cg))
mm = torch.sparse.mm

class DrugGNN(nn.Module):
    """Inductive drug encoder: a drug has NO embedding of its own — it is built from the genes it links to,
    so a never-seen drug gets a representation the same way a training drug does."""
    def __init__(s, ng, nd, d=64, drop=0.3):
        super().__init__()
        s.Eg, s.Ed = nn.Embedding(ng, d), nn.Embedding(nd, d)
        nn.init.normal_(s.Eg.weight, std=0.1); nn.init.normal_(s.Ed.weight, std=0.1)
        L = lambda i=d: nn.Linear(i, d)
        s.g_self, s.g_gg, s.g_gd = L(), L(), L()          # gene layer 1: itself + neighbour genes + linked diseases
        s.d_self, s.d_dg, s.d2 = L(), L(), L()            # disease layers: itself + linked genes
        s.c1, s.c2 = L(), L()                             # drug layers: from linked genes (layer 0 and layer 1)
        s.out_c, s.out_d = nn.Linear(2 * d, d), nn.Linear(3 * d, d)
        s.bias_d = nn.Parameter(torch.zeros(nd)); s.drop = nn.Dropout(drop)
    def encode(s, ops):
        g0, d0 = s.drop(s.Eg.weight), s.drop(s.Ed.weight)
        g1 = Fn.relu(s.g_self(g0) + s.g_gg(mm(ops["gg"], g0)) + s.g_gd(mm(ops["gd"], d0)))
        d1 = Fn.relu(s.d_self(d0) + s.d_dg(mm(ops["dg"], g0)))
        d2 = Fn.relu(s.d2(mm(ops["dg"], g1)))
        c1 = Fn.relu(s.c1(mm(ops["cg"], g0))); c2 = Fn.relu(s.c2(mm(ops["cg"], g1)))
        return s.out_c(torch.cat([c1, c2], 1)), s.out_d(torch.cat([d0, d1, d2], 1))
    def scores(s, zc, zd, rows=None):
        zc = zc if rows is None else zc[rows]
        return zc @ zd.T + s.bias_d

def train_fold(k, epochs=400, d=64, lr=0.01, wd=1e-4, drop=0.3, neg=5, seed=0, eval_every=10, patience=8, verbose=True):
    torch.manual_seed(seed); rg = np.random.default_rng(seed)
    f = folds[k]; tr_lab = np.setdiff1d(has, f)
    val = rg.choice(tr_lab, max(1, int(0.1 * len(tr_lab))), replace=False); trn = np.setdiff1d(tr_lab, val)
    Ytr = Y.copy(); Ytr[f] = 0; Ytr[val] = 0                              # test + validation drugs: labels hidden from training
    pr, pc = np.nonzero(Ytr); pr_t, pc_t = torch.tensor(pr, device=dev), torch.tensor(pc, device=dev)
    model = DrugGNN(ng, nd, d, drop).to(dev); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best, best_state, bad, t0 = -1, None, 0, time.time()
    def full_scores(rows):
        model.eval()
        with torch.no_grad():
            zc, zd = model.encode(OPS); return model.scores(zc, zd, torch.tensor(rows, device=dev)).cpu().numpy()
    def eval_rows(rows):
        S = np.zeros((nc, nd), np.float32); S[rows] = full_scores(rows)
        r, a = ranks_for(S, rows); r = np.array(r); return float(np.mean(1 / r)), float(np.mean(r <= 10))
    for ep in range(1, epochs + 1):
        model.train(); zc, zd = model.encode(OPS)
        nr = torch.tensor(rg.choice(trn, len(pr) * neg), device=dev); ncol = torch.tensor(rg.integers(0, nd, len(pr) * neg), device=dev)
        pos_logit = (zc[pr_t] * zd[pc_t]).sum(1) + model.bias_d[pc_t]
        neg_logit = (zc[nr] * zd[ncol]).sum(1) + model.bias_d[ncol]
        loss = Fn.binary_cross_entropy_with_logits(torch.cat([pos_logit, neg_logit]),
                                                   torch.cat([torch.ones_like(pos_logit), torch.zeros_like(neg_logit)]))
        opt.zero_grad(); loss.backward(); opt.step()
        if ep % eval_every == 0:
            vm, vh = eval_rows(val)
            if verbose: print(f"  ep {ep:3d} loss {loss.item():.4f} | val MRR {vm:.3f} Hits@10 {vh:.3f} | {time.time()-t0:.0f}s")
            if vm > best: best, bad, best_state = vm, 0, {n_: v.clone() for n_, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience: break
    model.load_state_dict(best_state)
    S = np.zeros((nc, nd), np.float32); S[f] = full_scores(f)
    return model, S, best

print("running fold 0 only, as a first check...")
m0, S0, v0 = train_fold(0)
r, a = ranks_for(S0, folds[0]); out0 = summarize("GNN fold 0", r, a)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in out0.items()})
rc, ac = ranks_for(sc_all[0], folds[0]); print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in summarize("Baseline C, same fold 0", rc, ac).items()})

In [ ]:
from sklearn.decomposition import TruncatedSVD

def gene_features(k=64):
    """Label-free structural summary of every gene: SVD of its connections to genes, diseases and drugs (+ log-degrees)."""
    dg = np.maximum(np.asarray(A_gg.sum(1)).ravel(), 1)
    M = sp.hstack([sp.diags(1 / np.sqrt(dg)) @ A_gg, A_gd, A_cg.T]).tocsr().astype(np.float32)
    nrm = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel()); nrm[nrm == 0] = 1
    Z = TruncatedSVD(min(k, M.shape[1] - 1), random_state=0).fit_transform(sp.diags(1 / nrm) @ M)
    deg = np.log1p(np.stack([np.asarray(X.sum(1)).ravel() for X in (A_gg, A_gd, A_cg.T)], 1))
    Z = np.hstack([Z, deg]); Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-6)
    return torch.tensor(Z, dtype=torch.float32)

t0 = time.time(); XG = gene_features().to(dev); print("gene features:", tuple(XG.shape), f"{time.time()-t0:.0f}s")

_Base = _Base if "_Base" in globals() else DrugGNN          # keep the v1 class so this cell can be re-run safely
class DrugGNN(_Base):
    """v2: genes start from structural features through ONE shared linear layer, instead of 39k free embeddings."""
    def __init__(s, ng, nd, d=64, drop=0.3):
        super().__init__(ng, nd, d, drop)
        s.gproj = nn.Linear(XG.shape[1], d); del s.Eg
    def encode(s, ops):
        g0, d0 = s.drop(s.gproj(XG)), s.drop(s.Ed.weight)
        g1 = Fn.relu(s.g_self(g0) + s.g_gg(mm(ops["gg"], g0)) + s.g_gd(mm(ops["gd"], d0)))
        d1 = Fn.relu(s.d_self(d0) + s.d_dg(mm(ops["dg"], g0)))
        d2 = Fn.relu(s.d2(mm(ops["dg"], g1)))
        c1 = Fn.relu(s.c1(mm(ops["cg"], g0))); c2 = Fn.relu(s.c2(mm(ops["cg"], g1)))
        return s.out_c(torch.cat([c1, c2], 1)), s.out_d(torch.cat([d0, d1, d2], 1))

print("training GNN v2 on fold 0...")
m1, S1, v1 = train_fold(0)
r1, a1 = ranks_for(S1, folds[0]); out1 = summarize("GNN v2 (structural genes), fold 0", r1, a1)
rc, ac = ranks_for(sc_all[0], folds[0]); outc = summarize("Baseline C, fold 0", rc, ac)
rows_ = [out0, out1, outc]
print(pd.DataFrame(rows_).round(3).to_string(index=False))

In [ ]:
# ---- 1) GNN v2 on all 5 folds (fold 0 already done) ----
gnn_S = {0: S1}
for k in range(1, 5):
    t0 = time.time(); _, Sk, vk = train_fold(k, verbose=False); gnn_S[k] = Sk
    print(f"fold {k} trained in {time.time()-t0:.0f}s | best val MRR {vk:.3f}")
R, A = [], []
for k, f in enumerate(folds):
    r, a = ranks_for(gnn_S[k], f); R += r; A += a
out_g = summarize("GNN v2 (5-fold)", R, A)

# ---- 2) blend GNN + baseline C: per-drug z-scores, weight chosen WITHOUT looking at the fold being scored ----
def rowz(S, rows):
    X = S[rows]; return (X - X.mean(1, keepdims=True)) / (X.std(1, keepdims=True) + 1e-9)
def logit(P):
    P = np.clip(P, 1e-6, 1 - 1e-6); return np.log(P / (1 - P))
def blend(k, w):
    f = folds[k]; S = np.zeros((nc, nd), np.float32)
    S[f] = w * rowz(logit(sc_all[k]), f) + (1 - w) * rowz(gnn_S[k], f)
    return S
ws = np.linspace(0, 1, 11)
fold_mrr = np.zeros((5, len(ws)))
for k, f in enumerate(folds):
    for j, w in enumerate(ws):
        r, _ = ranks_for(blend(k, w), f); fold_mrr[k, j] = np.mean(1 / np.array(r))
print("MRR by blend weight on baseline C (0 = GNN only, 1 = C only):")
print(pd.DataFrame(fold_mrr, columns=[f"{w:.1f}" for w in ws]).round(3).to_string())

R, A, chosen = [], [], []
for k, f in enumerate(folds):
    others = [i for i in range(5) if i != k]
    w = ws[np.argmax(fold_mrr[others].mean(0))]; chosen.append(round(float(w), 1))
    r, a = ranks_for(blend(k, w), f); R += r; A += a
out_b = summarize("Blend: GNN v2 + baseline C", R, A)
print("weights chosen per fold (never using the fold itself):", chosen)

tbl = pd.DataFrame([out_all, out_g, out_b]).round(3)
tbl.to_csv(WORK / "results_stage2.csv", index=False); print(); print(tbl.to_string(index=False))

In [ ]:
# ---- one out-of-fold score matrix per model: every labelled drug is scored by the fold model that NEVER saw it ----
def oof(get):
    M = np.zeros((nc, nd), np.float32)
    for k, f in enumerate(folds): M[f] = get(k)[f]
    return M
M_C = oof(lambda k: logit(sc_all[k])); M_G = oof(lambda k: gnn_S[k])
gz = lambda M: (M - M[has].mean()) / (M[has].std() + 1e-9)             # ONE global scale, so scores are comparable ACROSS drugs
M_B = 0.5 * gz(M_C) + 0.5 * gz(M_G)                                    # blend for the disease-first view (weight 0.5 = middle of the plateau)
M_R = rng.random((nc, nd)).astype(np.float32)

# ---- A) DISEASE-FIRST ranking: for each disease, rank the 1,324 labelled drugs; where do its true drugs land? ----
def disease_first(M, name):
    R, A = ranks_for(M[has].T, range(nd), Y[has].T)
    out = summarize(name, R, A); out["median_rank"] = float(np.median(R)); return out
tA = pd.DataFrame([disease_first(M_R, "random"), disease_first(M_C, "Baseline C"),
                   disease_first(M_G, "GNN v2"), disease_first(M_B, "Blend")]).round(3)
print(f"DISEASE-FIRST: rank among {len(has)} candidate drugs (so random median rank is about {len(has)//2})")
print(tA.drop(columns="Hits@50").to_string(index=False))

# ---- B) 95% confidence intervals for the DRUG-FIRST gains (resampling whole drugs) ----
M_Bd = oof(lambda k: blend(k, chosen[k]))                              # the per-drug blend used in the earlier table
def per_drug(M):
    rr, h10 = np.zeros(len(has)), np.zeros(len(has)); cnt = np.zeros(len(has))
    for i, d in enumerate(has):
        r = np.array(ranks_for(M, [d])[0]); rr[i], h10[i], cnt[i] = (1 / r).sum(), (r <= 10).sum(), len(r)
    return rr, h10, cnt
pdC, pdG, pdB = per_drug(M_C), per_drug(M_G), per_drug(M_Bd)
bi = rng.integers(0, len(has), (2000, len(has)))
def boot(a, b, j):                                                      # difference in a metric (b minus a), resampling drugs
    ma = a[j][bi].sum(1) / a[2][bi].sum(1); mb = b[j][bi].sum(1) / b[2][bi].sum(1); d = mb - ma
    return np.percentile(d, [2.5, 50, 97.5]), float((d > 0).mean())
print("\nDRUG-FIRST gains with 95% CI (2000 resamples of drugs):")
for nm_, (a, b) in {"Blend - Baseline C": (pdC, pdB), "GNN v2 - Baseline C": (pdC, pdG)}.items():
    for j, mn in [(0, "MRR"), (1, "Hits@10")]:
        (lo, mid, hi), p = boot(a, b, j)
        print(f"  {nm_:22s} {mn:8s} {mid:+.3f}  [{lo:+.3f}, {hi:+.3f}]  | resamples where it wins: {p:.0%}")

In [ ]:
from sklearn.decomposition import TruncatedSVD
# ---- 1) extra DRUG-side relations from the raw file (none involve diseases, so no label leakage) ----
raw = pd.read_csv(WORK / "drkg.tsv", sep="\t", header=None, names=["head", "rel", "tail"])
H_, T_ = raw["head"].values, raw["tail"].values
ht_, tt_ = raw["head"].str.split("::").str[0].values, raw["tail"].str.split("::").str[0].values
cm = {e: i for i, e in enumerate(comps)}
def block(T):
    m1 = (ht_ == "Compound") & (tt_ == T)
    if T == "Compound": a, b = H_[m1], T_[m1]
    else: m2 = (ht_ == T) & (tt_ == "Compound"); a, b = np.concatenate([H_[m1], T_[m2]]), np.concatenate([T_[m1], H_[m2]])
    r = pd.Series(a).map(cm); ok = r.notna().values
    if T == "Compound":
        c = pd.Series(b).map(cm); ok = ok & c.notna().values
        M = sp.csr_matrix((np.ones(ok.sum(), np.float32), (r[ok].astype(int), c[ok].astype(int))), shape=(nc, nc)); M = M + M.T
    else:
        u = {e: i for i, e in enumerate(sorted(set(b[ok])))}; c = pd.Series(b[ok]).map(u)
        M = sp.csr_matrix((np.ones(ok.sum(), np.float32), (r[ok].astype(int), c.astype(int))), shape=(nc, len(u)))
    M.data[:] = 1; return M.tocsr()
BL = {}
for T in ["Side Effect", "Compound"]:
    M = block(T); print(f"{T:20s} edges {M.nnz:>9,} | drugs covered {(M.sum(1) > 0).mean():.0%}")
    if M.nnz: BL[T] = M
del raw, H_, T_, ht_, tt_

def l2(M):
    M = sp.csr_matrix(M, dtype=np.float32); n = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel()); n[n == 0] = 1
    return (sp.diags(1 / n) @ M).tocsr()
idf = lambda M: M @ sp.diags(1 / np.log(2 + np.asarray(M.sum(0)).ravel()))
blocks = [A_cg] + list(BL.values())
SIMX = sum((lambda V: (V @ V.T).toarray().astype(np.float32))(l2(idf(M))) for M in blocks) / len(blocks)   # average cosine over all relation types
np.fill_diagonal(SIMX, 0)
Z = TruncatedSVD(64, random_state=0).fit_transform(l2(sp.hstack([idf(M) for M in BL.values()]).tocsr()))
XC = torch.tensor((Z - Z.mean(0)) / (Z.std(0) + 1e-6), dtype=torch.float32).to(dev)

# ---- 2) strong baseline again, with the richer drug similarity ----
dgd_ = np.asarray(A_gd.sum(0)).ravel().astype(np.float32); dgc_ = np.asarray(A_cg.sum(1)).ravel().astype(np.float32)
S2x = (A_cg @ A_gd).toarray().astype(np.float32); S3x = ((A_cg @ A_gg) @ A_gd).toarray().astype(np.float32)
SS_ = SIMX.sum(1) + 1e-6
def run_C2(label, seed=42):
    rg = np.random.default_rng(seed); R, A, sc = [], [], {}
    for k, f in enumerate(folds):
        Yt = Y_train_for(f); K = SIMX @ Yt
        F = np.stack([np.log1p(S2x), np.log1p(S3x), S2x / np.sqrt(1 + dgd_)[None, :], K, K / SS_[:, None], np.broadcast_to(Yt.sum(0), (nc, nd)),
                      np.broadcast_to(np.log1p(dgc_)[:, None], (nc, nd)), np.broadcast_to(np.log1p(dgd_)[None, :], (nc, nd))], -1).astype(np.float32)
        tl = np.setdiff1d(has, f); pr, pcol = np.nonzero(Yt[tl]); pr = tl[pr]
        nr, ncl = rg.choice(tl, len(pr) * 5), rg.integers(0, nd, len(pr) * 5); kp = Yt[nr, ncl] == 0
        X = np.concatenate([F[pr, pcol], F[nr[kp], ncl[kp]]]); y = np.concatenate([np.ones(len(pr)), np.zeros(kp.sum())])
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_depth=5, early_stopping=True, validation_fraction=0.15, random_state=seed).fit(X, y)
        Sf = np.zeros((nc, nd), np.float32); Sf[f] = m.predict_proba(F[f].reshape(-1, F.shape[-1]))[:, 1].reshape(len(f), nd)
        sc[k] = Sf; r, a = ranks_for(Sf, f); R += r; A += a
    return summarize(label, R, A), sc
outC2, sc_v3 = run_C2("C2. baseline + richer drug similarity"); print({k: round(v, 3) for k, v in outC2.items() if k not in ("model",)})

# ---- 3) GNN v3: v2 + a label-free structural descriptor of each drug from the extra relations ----
_V2 = _V2 if "_V2" in globals() else DrugGNN
class DrugGNN(_V2):
    def __init__(s, ng, nd, d=64, drop=0.3):
        super().__init__(ng, nd, d, drop); s.cproj = nn.Linear(XC.shape[1], d)
    def encode(s, ops):
        zc, zd = super().encode(ops); return zc + s.cproj(s.drop(XC)), zd
gnn_v3 = {}
for k in range(5): _, gnn_v3[k], _ = train_fold(k, verbose=False)
R, A = [], []
for k, f in enumerate(folds): r, a = ranks_for(gnn_v3[k], f); R += r; A += a
outG3 = summarize("GNN v3 (+ extra relations)", R, A)

# ---- 4) blend (weight chosen without the scored fold) ----
def bl(Sc, Sg, k, w):
    f = folds[k]; S = np.zeros((nc, nd), np.float32); S[f] = w * rowz(logit(Sc[k]), f) + (1 - w) * rowz(Sg[k], f); return S
fm = np.array([[np.mean(1 / np.array(ranks_for(bl(sc_v3, gnn_v3, k, w), folds[k])[0])) for w in ws] for k in range(5)])
R, A = [], []
for k, f in enumerate(folds):
    w = ws[np.argmax(fm[[i for i in range(5) if i != k]].mean(0))]; r, a = ranks_for(bl(sc_v3, gnn_v3, k, w), f); R += r; A += a
outB3 = summarize("Blend v3", R, A)
tbl3 = pd.DataFrame([out_all, out_g, out_b, outC2, outG3, outB3]).round(3); tbl3.to_csv(WORK / "results_v3.csv", index=False); print(tbl3.to_string(index=False))

In [ ]:
import glob, urllib.request, shutil
def fetch(url, dst):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 Chrome/124"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dst, "wb") as f: shutil.copyfileobj(r, f, 1 << 20)
def parse_mesh(path):
    out, mh, ui = {}, None, None
    for line in open(path, encoding="utf-8", errors="ignore"):
        if line.startswith("*NEWRECORD"):
            if mh and ui: out[f"Disease::MESH:{ui}"] = mh
            mh = ui = None
        elif line.startswith("MH = "): mh = line[5:].strip()
        elif line.startswith("UI = "): ui = line[5:].strip()
    if mh and ui: out[f"Disease::MESH:{ui}"] = mh
    return out
name = {}
hp = WORK / "hetionet-v1.0-nodes.tsv"                                   # gene symbols + drug names (CC0)
if not hp.exists(): fetch("https://raw.githubusercontent.com/hetio/hetionet/main/hetnet/tsv/hetionet-v1.0-nodes.tsv", hp)
hn = pd.read_csv(hp, sep="\t"); name.update({i: n_ for i, n_, k in hn[hn.kind.isin(["Gene", "Compound"])].itertuples(index=False)})
for p in glob.glob("/kaggle/input/**/*", recursive=True):               # MeSH file you uploaded (d20xx.bin)
    if Path(p).is_file() and Path(p).stat().st_size > 5e6:
        if "MH = " in open(p, encoding="utf-8", errors="ignore").read(200000): name.update(parse_mesh(p)); print("MeSH loaded from", p); break
print("names:", len(name), "| diseases named:", sum(k.startswith("Disease::") for k in name), "of", nd)
nm = lambda e: name.get(e, e.split("::", 1)[-1])

# ---- evidence ranked by informativeness: genes linked to FEW diseases/drugs count most (IDF weighting) ----
wg = 1 / np.log(2 + np.asarray(A_gd.sum(1)).ravel()) / np.log(2 + np.asarray(A_cg.sum(0)).ravel())
def evidence(d, x, top=3):
    two = A_cg[d].multiply(A_gd[:, x].T).tocsr().indices; two = two[np.argsort(-wg[two])][:top]
    dg, xg = A_cg[d].indices, A_gd[:, x].nonzero()[0]; sub = A_gg[dg][:, xg].tocoo()
    pr = sorted(zip(sub.row, sub.col), key=lambda p: -wg[dg[p[0]]] * wg[xg[p[1]]])[:top]
    return "direct genes: " + (", ".join(nm(genes[g]) for g in two) or "-") + " | via interacting genes: " + (", ".join(f"{nm(genes[dg[i]])}~{nm(genes[xg[j]])}" for i, j in pr) or "-")

def retro(drug_id, S_oof, topk=5):
    e = f"Compound::{drug_id}"
    if e not in cmap_: print(drug_id, "not in graph"); return
    d = cmap_[e]
    if d not in set(has): print(f"{nm(e)}: no DrugBank 'treats' label"); return
    S = S_oof[d]; order = np.argsort(-S); rank = np.empty(nd, int); rank[order] = np.arange(1, nd + 1)
    print(f"\n=== {nm(e)} — TRUE uses and model rank (of {nd}) ===")
    for x in np.where(Y[d] > 0)[0]: print(f"  rank {rank[x]:4d}  {nm(diseases[x])}\n            {evidence(d, x)}")
    print("  top suggestions:", "; ".join(f"{nm(diseases[x])}{'*' if Y[d, x] else ''}" for x in order[:topk]), "(* = known use)")
cmap_ = {e: i for i, e in enumerate(comps)}
S_best = np.zeros((nc, nd), np.float32)
for k, f in enumerate(folds): S_best[f] = bl(sc_v3, gnn_v3, k, 0.5)[f]            # out-of-fold scores from the v3 blend
for dr in ["DB01041", "DB00571", "DB00481", "DB00331", "DB00203"]: retro(dr, S_best)

In [ ]:
#cell F
# ---- STEP 2: confidence intervals for the final (ablated) blend; drug-first, resampling whole drugs ----
M_V3 = oof(lambda k: bl(sc_v3, gnn_v3, k, 0.5)); pdV3 = per_drug(M_V3)
print("DRUG-FIRST gains, 95% CI (2000 resamples of drugs):")
for nm_, (a, b) in {"Blend v3 - Baseline C": (pdC, pdV3), "Blend v3 - Blend v2": (pdB, pdV3)}.items():
    for j, mn in [(0, "MRR"), (1, "Hits@10")]:
        (lo, mid, hi), p = boot(a, b, j); print(f"  {nm_:24s} {mn:8s} {mid:+.3f} [{lo:+.3f}, {hi:+.3f}] | wins {p:.0%}")

# ---- STEP 3: final models trained on ALL labelled drugs, then a score for EVERY drug x disease ----
folds.append(np.array([], dtype=int)); gm, _, _ = train_fold(len(folds) - 1, verbose=False); folds.pop()   # empty test fold = train on everything
gm.eval()
with torch.no_grad(): zc_, zd_ = gm.encode(OPS); G_all = gm.scores(zc_, zd_).cpu().numpy()
F = np.stack([np.log1p(S2x), np.log1p(S3x), S2x / np.sqrt(1 + dgd_)[None, :], SIMX @ Y, (SIMX @ Y) / SS_[:, None], np.broadcast_to(Y.sum(0), (nc, nd)),
              np.broadcast_to(np.log1p(dgc_)[:, None], (nc, nd)), np.broadcast_to(np.log1p(dgd_)[None, :], (nc, nd))], -1).astype(np.float32)
rg = np.random.default_rng(42); pr, pcol = np.nonzero(Y[has]); pr = has[pr]
nr, ncl = rg.choice(has, len(pr) * 5), rg.integers(0, nd, len(pr) * 5); kp = Y[nr, ncl] == 0
X = np.concatenate([F[pr, pcol], F[nr[kp], ncl[kp]]]); yy = np.concatenate([np.ones(len(pr)), np.zeros(kp.sum())])
mC = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_depth=5, early_stopping=True, validation_fraction=0.15, random_state=42).fit(X, yy)
P_all = mC.predict_proba(F.reshape(-1, F.shape[-1]))[:, 1].reshape(nc, nd); del F, X
zf = lambda M: (M - M.mean()) / (M.std() + 1e-9)
FINAL = (0.5 * zf(logit(P_all)) + 0.5 * zf(G_all)).astype(np.float32)
np.save(WORK / "final_scores_v3_ablated.npy", FINAL); pd.to_pickle(dict(comps=comps, diseases=diseases), WORK / "final_index.pkl")
print("saved final scores:", FINAL.shape, "-> final_scores_v3_ablated.npy")

In [ ]:
# ---- Cell I: 3-seed GNN ensemble (seed 0 = existing models); used only if it beats the single model ----
ens = {k: [gnn_v3[k]] for k in range(5)}
for sd in (1, 2):
    for k in range(5): ens[k].append(train_fold(k, seed=sd, verbose=False)[1])
gnn_e = {k: np.mean(ens[k], 0) for k in range(5)}
def loo_blend(Sg):
    fm = np.array([[np.mean(1 / np.array(ranks_for(bl(sc_v3, Sg, k, w), folds[k])[0])) for w in ws] for k in range(5)])
    R, A = [], []
    for k, f in enumerate(folds):
        w = ws[np.argmax(fm[[i for i in range(5) if i != k]].mean(0))]; r, a = ranks_for(bl(sc_v3, Sg, k, w), f); R += r; A += a
    return R, A
R, A = [], []
for k, f in enumerate(folds): r, a = ranks_for(gnn_e[k], f); R += r; A += a
outGe = summarize("GNN x3 seeds", R, A); outBe = summarize("Blend v3 x3 seeds", *loo_blend(gnn_e))
tbl_e = pd.DataFrame([outG3, outGe, outB3, outBe]).round(3); tbl_e.to_csv(WORK / "results_ensemble.csv", index=False); print(tbl_e.to_string(index=False))

# final models on ALL labelled drugs for seeds 1,2 (seed 0 = G_all from Cell F)
Gs = [G_all]; folds.append(np.array([], dtype=int))
for sd in (1, 2):
    gm_, _, _ = train_fold(len(folds) - 1, seed=sd, verbose=False); gm_.eval()
    with torch.no_grad(): z1, z2 = gm_.encode(OPS); Gs.append(gm_.scores(z1, z2).cpu().numpy())
folds.pop()
if outBe["MRR"] >= outB3["MRR"]:
    FINAL = (0.5 * zf(logit(P_all)) + 0.5 * zf(np.mean(Gs, 0))).astype(np.float32); print("ensemble ADOPTED")
else: print("ensemble not better -> keeping single-model FINAL")
np.save(WORK / "final_scores_v4.npy", FINAL)

In [ ]:
#cell G
import json
from collections import defaultdict
# ---- safety info (DRKG side effects + drug-drug interactions) ----
hn = pd.read_csv(WORK / "hetionet-v1.0-nodes.tsv", sep="\t"); name.update({i: n_ for i, n_, k in hn[hn.kind == "Side Effect"].itertuples(index=False)})
raw = pd.read_csv(WORK / "drkg.tsv", sep="\t", header=None, names=["head", "rel", "tail"])
H_, T_ = raw["head"].values, raw["tail"].values
ht_, tt_ = raw["head"].str.split("::").str[0].values, raw["tail"].str.split("::").str[0].values
SE, DDI = defaultdict(set), defaultdict(set)
m1 = (ht_ == "Compound") & (tt_ == "Side Effect"); m2 = (ht_ == "Side Effect") & (tt_ == "Compound")
for h, t in zip(H_[m1], T_[m1]): SE[h].add(t)
for h, t in zip(H_[m2], T_[m2]): SE[t].add(h)
for h, t in zip(H_[(ht_ == "Compound") & (tt_ == "Compound")], T_[(ht_ == "Compound") & (tt_ == "Compound")]): DDI[h].add(t); DDI[t].add(h)
SERIOUS = ("hepat", "liver", "cardiac arrest", "arrhythm", "torsade", "death", "suicid", "renal failure", "kidney failure", "bleed", "hemorrh", "haemorrh",
           "anaphyla", "agranulocyt", "seizure", "convuls", "stroke", "thrombo", "embol", "pancreatitis", "stevens", "necrolysis", "leukopenia", "neutropenia", "aplastic", "psychosis", "infarct")
def safety(e):
    s, n = SE.get(e), len(DDI.get(e, ()))
    if s:
        flag = sorted({nm(x) for x in s if any(k in nm(x).lower() for k in SERIOUS)})[:4]
        txt = f"{len(s)} reported side effects" + (" incl. " + ", ".join(flag) if flag else "")
    else: txt = "no side-effect data"
    if DEG.get(e, 0) >= T95: txt += " | WARNING: unusually many gene links, score may be inflated"
    return txt + (f" | interacts with {n} other drugs" if n else "")
print("side-effect names found:", sum(k.startswith("Side Effect::") for k in name))

pool = np.array([i for i, e in enumerate(comps) if e in name] or list(range(nc)))
print(f"candidate drugs shown: {len(pool)} of {nc}")
# flag drugs with unusually many gene links (top 5% of candidates): such compounds tend to be over-ranked
DEG = dict(zip(comps, np.asarray(A_cg.sum(1)).ravel())); T95 = np.percentile([DEG[comps[i]] for i in pool], 95)
# evidence without unnamed genes (non-human IDs show up as bare numbers)
named = np.array([not nm(g).isdigit() for g in genes])
def evidence(d, x, top=3):
    two = np.array([g for g in A_cg[d].multiply(A_gd[:, x].T).tocsr().indices if named[g]], dtype=int); two = two[np.argsort(-wg[two])][:top]
    dg = np.array([g for g in A_cg[d].indices if named[g]], dtype=int); xg = np.array([g for g in A_gd[:, x].nonzero()[0] if named[g]], dtype=int)
    pr = []
    if len(dg) and len(xg):
        sub = A_gg[dg][:, xg].tocoo(); pr = sorted(zip(sub.row, sub.col), key=lambda p: -wg[dg[p[0]]] * wg[xg[p[1]]])[:top]
    return "direct genes: " + (", ".join(nm(genes[g]) for g in two) or "-") + " | via interacting genes: " + (", ".join(f"{nm(genes[dg[i]])}~{nm(genes[xg[j]])}" for i, j in pr) or "-")
zP = zf(logit(P_all)); SD = np.stack([0.5 * zP + 0.5 * zf(g) for g in Gs]).std(0).astype(np.float32)   # spread across the 3 seeds
TOP, TOPD = 25, 12; dis, drg, SAFE = {}, {}, {}
for x in range(nd):
    order = pool[np.argsort(-FINAL[pool, x])][:TOP]
    dis[nm(diseases[x])] = dict(id=diseases[x].split("::")[-1], known=int(Y[pool, x].sum()),
        rows=[[r + 1, nm(comps[d]), comps[d].split("::")[1], round(float(FINAL[d, x]), 2), bool(Y[d, x]), evidence(d, x), "", round(float(SD[d, x]), 2)] for r, d in enumerate(order)])
for d in pool:
    o = np.argsort(-FINAL[d])[:TOPD]; k = nm(comps[d]); SAFE[k] = safety(comps[d])
    drg[k] = dict(id=comps[d].split("::")[1], rows=[[r + 1, nm(diseases[x]), diseases[x].split("::")[-1].replace("MESH:", ""), round(float(FINAL[d, x]), 2), bool(Y[d, x]), evidence(d, x), "", round(float(SD[d, x]), 2)] for r, x in enumerate(o)])
for v in dis.values():
    for r in v["rows"]: r[6] = SAFE.get(r[1], "")
page = """<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Drug Repurposing Finder</title><style>
:root{--bg:#fff;--fg:#1c1e21;--mut:#667;--line:#e3e6ea;--acc:#1d6fb8;--ok:#e8f5e9}
@media(prefers-color-scheme:dark){:root{--bg:#14171a;--fg:#e8eaed;--mut:#9aa;--line:#2c3238;--acc:#6db3f2;--ok:#1d3a24}}
body{background:var(--bg);color:var(--fg);font:15px/1.5 system-ui,sans-serif;margin:0;padding:20px 16px;max-width:1100px;margin:auto}
h1{font-size:22px;margin:0 0 4px}.mut{color:var(--mut);font-size:13px}input,select{font:inherit;padding:8px;border:1px solid var(--line);border-radius:6px;background:var(--bg);color:var(--fg);width:100%;box-sizing:border-box;margin:6px 0}
table{border-collapse:collapse;width:100%;margin-top:12px}th,td{border-bottom:1px solid var(--line);padding:8px 6px;text-align:left;vertical-align:top;font-size:14px}
tr.k{background:var(--ok)}.ev{color:var(--mut);font-size:12px}.bar{height:6px;background:var(--acc);border-radius:3px}a{color:var(--acc)}button{font:inherit;padding:6px 12px;border:1px solid var(--line);border-radius:6px;background:var(--bg);color:var(--fg);cursor:pointer}.warn{margin-top:16px;font-size:12px;color:var(--mut);border-top:1px solid var(--line);padding-top:10px}
@media(max-width:640px){.ev{display:none}}</style></head><body>
<h1>Drug Repurposing Finder</h1><div class="mut">Search by disease (ranked existing drugs) or by drug (ranked diseases), with the biological evidence, safety info and a literature search link for each suggestion.</div>
<select id="mode"><option value="dis">Search by disease</option><option value="drg">Search by drug</option></select>
<input id="q" placeholder="Search..."><select id="s" size="1"></select><div id="meta" class="mut"></div><button id="dl">Download CSV</button><table><thead id="h"></thead><tbody id="t"></tbody></table>
<div class="warn"><b>Research leads only, not medical advice.</b> Scores are the model's relative scores (higher = more plausible), not probabilities. Shaded rows are uses already listed in DrugBank. Unlisted high-ranked suggestions may be genuine opportunities or false positives; an expert must verify. Safety text is a summary of reported side effects (SIDER via DRKG) and listed drug interactions, not a complete safety profile. The ± value is the spread across 3 model runs (training randomness), not a statistical confidence interval. Data: DRKG (Apache-2.0; underlying sources have their own licences, e.g. DrugBank CC BY-NC 4.0).</div>
<script>const D=__DATA__,SAFE=__SAFE__;let mode='dis';const $=i=>document.getElementById(i),s=$('s'),q=$('q'),t=$('t'),m=$('meta'),h=$('h'),md=$('mode');
const esc=x=>String(x).replace(/[&<>]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;'}[c]));
const pm=(a,b)=>'<a target="_blank" rel="noopener" href="https://pubmed.ncbi.nlm.nih.gov/?term='+encodeURIComponent('"'+a+'" AND "'+b+'"')+'">PubMed</a>';
function fill(){const f=q.value.toLowerCase();s.innerHTML=Object.keys(D[mode]).sort().filter(n=>n.toLowerCase().includes(f)).map(n=>'<option>'+esc(n)+'</option>').join('');s.size=Math.min(6,Math.max(1,s.options.length));if(s.options.length)s.selectedIndex=0;show()}
function show(){const d=D[mode][s.value],dm=mode=='dis';h.innerHTML='<tr><th>#</th><th>'+(dm?'Drug':'Disease')+'</th><th>Model score (± spread)</th><th>Evidence (genes)</th><th>Literature</th></tr>';
if(!d){t.innerHTML='';m.textContent='';return}
m.textContent=dm?'MeSH '+d.id.replace('MESH:','')+' | '+d.known+' drugs with a known DrugBank use (shaded)':'DrugBank '+d.id+' | Safety: '+SAFE[s.value];
const hi=d.rows[0][3],lo=d.rows[d.rows.length-1][3];t.innerHTML=d.rows.map(r=>'<tr class="'+(r[4]?'k':'')+'"><td>'+r[0]+'</td><td><b>'+esc(r[1])+'</b><div class="mut">'+esc(r[2])+(r[4]?' - known use':'')+'</div>'+(dm&&r[6]?'<div class="ev">Safety: '+esc(r[6])+'</div>':'')+'</td><td>'+r[3]+' <span class="ev">±'+r[7]+'</span><div class="bar" style="width:'+Math.max(4,100*(r[3]-lo)/(hi-lo+1e-9))+'%"></div></td><td class="ev">'+esc(r[5])+'</td><td>'+(dm?pm(r[1],s.value):pm(s.value,r[1]))+'</td></tr>').join('')}
$('dl').onclick=()=>{const d=D[mode][s.value];if(!d)return;const dm=mode=='dis',e=x=>'"'+String(x).replace(/"/g,'""')+'"';const csv=[['rank',dm?'drug':'disease','id','score','seed_spread','known_use','evidence'].join(',')].concat(d.rows.map(r=>[r[0],e(r[1]),e(r[2]),r[3],r[7],r[4],e(r[5])].join(','))).join('\\n');const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([csv],{type:'text/csv'}));a.download=s.value.replace(/[^A-Za-z0-9]+/g,'_')+'.csv';a.click()};
q.oninput=fill;s.onchange=show;md.onchange=()=>{mode=md.value;q.value='';fill()};fill();</script></body></html>"""
out = WORK / "dashboard.html"
out.write_text(page.replace("__DATA__", json.dumps(dict(dis=dis, drg=drg)).replace("</", "<\\/")).replace("__SAFE__", json.dumps(SAFE).replace("</", "<\\/")), encoding="utf-8")
print("dashboard saved:", out, f"({out.stat().st_size/1e6:.1f} MB)")

In [ ]:
# ===== Cell J: PRO dashboard (run AFTER Cell G; overwrites dashboard.html; Cell H then uploads it) =====
import json
zB = zf(logit(P_all)); zG = zf(np.mean(Gs, 0))      # the two models' scores (boosting, graph network)
_ev = {}
def ev_s(d, x, top=3):                              # evidence as data: direct genes + interacting gene pairs (named genes only)
    if (d, x) in _ev: return _ev[(d, x)]
    two = np.array([g for g in A_cg[d].multiply(A_gd[:, x].T).tocsr().indices if named[g]], dtype=int); two = two[np.argsort(-wg[two])][:top]
    dg = np.array([g for g in A_cg[d].indices if named[g]], dtype=int); xg = np.array([g for g in A_gd[:, x].nonzero()[0] if named[g]], dtype=int)
    pr = []
    if len(dg) and len(xg):
        sub = A_gg[dg][:, xg].tocoo(); pr = sorted(zip(sub.row, sub.col), key=lambda p: -wg[dg[p[0]]] * wg[xg[p[1]]])[:top]
    _ev[(d, x)] = dict(d=[nm(genes[g]) for g in two], v=[[nm(genes[dg[i]]), nm(genes[xg[j]])] for i, j in pr]); return _ev[(d, x)]
def row(r, d, x, nameval, idval, sf=""):
    return [r + 1, nameval, idval, round(float(FINAL[d, x]), 2), bool(Y[d, x]), ev_s(d, x), sf, round(float(SD[d, x]), 2), round(float(zB[d, x]), 1), round(float(zG[d, x]), 1)]
TOP, TOPD = 25, 12; dis, drg = {}, {}
for x in range(nd):
    order = pool[np.argsort(-FINAL[pool, x])][:TOP]
    dis[nm(diseases[x])] = dict(id=diseases[x].split("::")[-1].replace("MESH:", ""), known=int(Y[pool, x].sum()),
        rows=[row(r, d, x, nm(comps[d]), comps[d].split("::")[1], SAFE.get(nm(comps[d]), "")) for r, d in enumerate(order)])
for d in pool:
    o = np.argsort(-FINAL[d])[:TOPD]
    drg[nm(comps[d])] = dict(id=comps[d].split("::")[1], rows=[row(r, d, x, nm(diseases[x]), diseases[x].split("::")[-1].replace("MESH:", "")) for r, x in enumerate(o)])
Sc = FINAL[pool].astype(np.float64).copy(); Sc[Y[pool] == 1] = -1e9; K = min(300, Sc.size - 1)     # discovery feed: best-scored pairs NOT already known
idx = np.argpartition(-Sc.ravel(), K)[:K]; idx = idx[np.argsort(-Sc.ravel()[idx])]
FEED = []
for r, i in enumerate(idx):
    d, x = pool[i // nd], i % nd; FEED.append(row(r, d, x, nm(comps[d]), comps[d].split("::")[1], SAFE.get(nm(comps[d]), "")) + [nm(diseases[x])])
META = dict(nc=int(nc), nd=int(nd), ng=int(len(genes)), pool=int(len(pool)), pairs=int(nc) * int(nd), labels=int(Y.sum()), labdrugs=int((Y.sum(1) > 0).sum()))
page = r"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Drug Repurposing Finder</title><style>
:root{--bg:#f5f2ea;--pn:#fbf9f3;--pn2:#ece6d6;--ln:#d9d2c0;--fg:#14262b;--mu:#566a6e;--ac:#1e7a72;--ac2:#c23b20;--ok:#dcebe8;--am:#a9650b;--sh:0 6px 24px rgba(20,38,43,.10)}
@media(prefers-color-scheme:dark){:root:not([data-theme=light]){--bg:#0b1c21;--pn:#10272e;--pn2:#173842;--ln:#25484f;--fg:#eaf1ef;--mu:#9db4b6;--ac:#7fd1c4;--ac2:#f28b6a;--ok:#16382f;--am:#f0b35a;--sh:0 6px 24px rgba(0,0,0,.4)}}
:root[data-theme=dark]{--bg:#0b1c21;--pn:#10272e;--pn2:#173842;--ln:#25484f;--fg:#eaf1ef;--mu:#9db4b6;--ac:#7fd1c4;--ac2:#f28b6a;--ok:#16382f;--am:#f0b35a;--sh:0 6px 24px rgba(0,0,0,.4)}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--fg);font:15px/1.5 "IBM Plex Sans",system-ui,-apple-system,Segoe UI,sans-serif}
h1,h2,h3{font-family:Georgia,"Fraunces",serif;margin:0;line-height:1.15}a{color:var(--ac)}button{font:inherit;color:inherit;cursor:pointer}
.app{display:grid;grid-template-columns:232px 1fr;min-height:100vh}
.side{background:var(--pn);border-right:1px solid var(--ln);padding:18px 12px;position:sticky;top:0;height:100vh;display:flex;flex-direction:column;gap:4px;overflow:auto}
.brand{display:flex;gap:10px;align-items:center;padding:4px 8px 14px}.logo{width:34px;height:34px;border-radius:10px;background:var(--ac);display:grid;place-items:center;color:var(--bg);font-weight:700}
.brand b{font-family:Georgia,serif;font-size:17px;line-height:1.1}.nav{display:flex;gap:10px;align-items:center;border:0;background:none;text-align:left;padding:9px 10px;border-radius:10px;color:var(--mu);font-weight:500}
.nav:hover{background:var(--pn2);color:var(--fg)}.nav.on{background:var(--ac);color:var(--bg)}.nav i{font-style:normal;width:20px;text-align:center}.nav .ct{margin-left:auto;background:var(--ac2);color:#fff;border-radius:9px;padding:0 7px;font-size:12px}
.sp{flex:1}.main{padding:18px 26px 40px;min-width:0}.top{display:flex;gap:12px;align-items:center;margin-bottom:18px;position:relative}
.gs{flex:1;position:relative}.gs input{width:100%;padding:11px 14px 11px 38px;border-radius:12px;border:1px solid var(--ln);background:var(--pn);color:var(--fg);font:inherit}.gs:before{content:"\2315";position:absolute;left:13px;top:8px;font-size:19px;color:var(--mu)}
.sug{position:absolute;left:0;right:0;top:46px;background:var(--pn);border:1px solid var(--ln);border-radius:12px;box-shadow:var(--sh);z-index:30;overflow:hidden}.sug button{display:flex;gap:10px;width:100%;border:0;background:none;padding:9px 14px;text-align:left}.sug button:hover{background:var(--pn2)}
.tag{font-size:11px;letter-spacing:.06em;text-transform:uppercase;border-radius:6px;padding:1px 7px;background:var(--pn2);color:var(--mu);align-self:center}.ib{border:1px solid var(--ln);background:var(--pn);border-radius:12px;padding:10px 14px}
.card{background:var(--pn);border:1px solid var(--ln);border-radius:16px;padding:18px}.grid{display:grid;gap:14px}.g2{grid-template-columns:repeat(2,1fr)}.g3{grid-template-columns:repeat(3,1fr)}.g4{grid-template-columns:repeat(4,1fr)}.g5{grid-template-columns:repeat(5,1fr)}
.mu{color:var(--mu)}.sm{font-size:13px}.eyebrow{font-size:12px;font-weight:600;letter-spacing:.14em;text-transform:uppercase;color:var(--ac2)}
.hero{background:linear-gradient(135deg,#0f2a30,#173f47);color:#f5f2ea;border-radius:22px;padding:34px;display:grid;grid-template-columns:1.1fr .9fr;gap:20px;overflow:hidden;position:relative;border:1px solid var(--ln)}.hero h1{font-size:40px;margin:8px 0 12px}.hero p{color:#cfe0dd;max-width:560px}
.stat b{display:block;font-family:Georgia,serif;font-size:30px;color:var(--ac)}.stat span{font-size:13px;color:var(--mu)}.btn{border:1px solid var(--ln);background:var(--pn2);border-radius:10px;padding:8px 14px;font-weight:500}.btn:hover{border-color:var(--ac)}.btn.pri{background:var(--ac2);border-color:var(--ac2);color:#fff}
.chip{display:inline-block;border-radius:999px;padding:2px 10px;font-size:12px;background:var(--pn2);border:1px solid var(--ln);margin:2px 4px 2px 0;color:var(--fg)}.chip.g{background:var(--ok);border-color:var(--ac)}.chip.r{border-color:var(--ac2);color:var(--ac2)}.chip.a{border-color:var(--am);color:var(--am)}.chip.btn2{cursor:pointer}.chip.btn2:hover{border-color:var(--ac)}
.pick{display:grid;grid-template-columns:320px 1fr;gap:16px;align-items:start}.list{max-height:330px;overflow:auto;border:1px solid var(--ln);border-radius:12px;margin-top:8px}.list button{display:flex;justify-content:space-between;width:100%;border:0;border-bottom:1px solid var(--ln);background:none;padding:8px 12px;text-align:left}.list button:hover{background:var(--pn2)}.list button.on{background:var(--ac);color:var(--bg)}
input[type=text],input[type=search],select{padding:9px 12px;border-radius:10px;border:1px solid var(--ln);background:var(--bg);color:var(--fg);font:inherit;width:100%}
.flt{display:flex;flex-wrap:wrap;gap:14px;align-items:center;margin:12px 0}.flt label{display:flex;gap:6px;align-items:center;font-size:13px}
table{border-collapse:collapse;width:100%}th{font-size:12px;text-transform:uppercase;letter-spacing:.07em;color:var(--mu);text-align:left;padding:8px 8px;border-bottom:2px solid var(--ln)}td{padding:10px 8px;border-bottom:1px solid var(--ln);vertical-align:top}tr.rw{cursor:pointer}tr.rw:hover{background:var(--pn2)}tr.k td:first-child{box-shadow:inset 4px 0 var(--ac)}
.bar{height:7px;border-radius:4px;background:var(--ac);margin-top:5px}.star{border:0;background:none;font-size:20px;color:var(--mu)}.star.on{color:var(--am)}
.drawer{position:fixed;top:0;right:0;height:100vh;width:500px;max-width:100vw;background:var(--pn);border-left:1px solid var(--ln);box-shadow:var(--sh);z-index:50;transform:translateX(105%);transition:transform .25s;overflow:auto;padding:20px}.drawer.open{transform:none}
.tabs{display:flex;gap:4px;border-bottom:1px solid var(--ln);margin:14px 0}.tabs button{border:0;background:none;padding:8px 12px;color:var(--mu);border-bottom:3px solid transparent;font-weight:500}.tabs button.on{color:var(--fg);border-color:var(--ac2)}
.evg text{font-size:12px;fill:var(--fg);font-family:inherit}.evg .nd rect{fill:var(--pn2);stroke:var(--ln)}.evg .drug rect{fill:var(--ac2);stroke:none}.evg .drug text,.evg .dz text{fill:#fff;font-weight:600}.evg .dz rect{fill:var(--ac);stroke:none}.evg .gn rect{fill:var(--ok);stroke:var(--ac)}.evg line{stroke:var(--mu);stroke-width:1.6}.evg line.dsh{stroke-dasharray:5 4}
.pipe{display:flex;gap:8px;flex-wrap:wrap}.pipe button{flex:1;min-width:120px;border:1px solid var(--ln);background:var(--pn);border-radius:14px;padding:12px;text-align:left}.pipe button.on{background:var(--ac2);border-color:var(--ac2);color:#fff}.pipe b{font-family:Georgia,serif;font-size:22px;display:block}
details{border:1px solid var(--ln);border-radius:12px;padding:10px 14px;background:var(--pn)}summary{cursor:pointer;font-weight:600}
.toast{position:fixed;bottom:22px;left:50%;transform:translateX(-50%);background:var(--fg);color:var(--bg);padding:9px 18px;border-radius:12px;z-index:90;opacity:0;transition:opacity .2s;pointer-events:none}.toast.on{opacity:1}
.shade{position:fixed;inset:0;background:rgba(0,0,0,.35);z-index:40;display:none}.shade.on{display:block}
.bt{display:flex;height:22px;border-radius:6px;background:var(--pn2);overflow:hidden}.bt i{display:block;height:100%}
@keyframes pu{0%,100%{opacity:.35}50%{opacity:1}}.pu{animation:pu 3s infinite}@keyframes fl{to{stroke-dashoffset:-40}}.fl{stroke-dasharray:6 6;animation:fl 2.4s linear infinite}
@media(max-width:900px){.app{grid-template-columns:1fr}.side{position:static;height:auto;flex-direction:row;overflow:auto;padding:8px}.brand,.sp,.side .mu{display:none}.side{align-items:center}.nav{padding:8px 12px}.nav span{display:none}.hero,.pick,.g2,.g3,.g4,.g5{grid-template-columns:1fr}.main{padding:14px}.hero h1{font-size:30px}.hide-m{display:none}}
</style></head><body><div class="app"><nav class="side" id="nav"></nav><div class="main"><div class="top"><div class="gs"><input id="gs" type="search" placeholder="Search any disease or drug   ( press / )" autocomplete="off"><div class="sug" id="sug" hidden></div></div><button class="ib" id="th" title="Light / dark">Theme</button></div><div id="v"></div>
<div class="mu sm" style="margin-top:30px;border-top:1px solid var(--ln);padding-top:12px"><b>Research leads only, not medical advice.</b> Scores are relative (higher = more plausible), not probabilities. Shaded rows are uses already listed in DrugBank; unlisted high-ranked suggestions may be real opportunities or false positives and need expert checking. Safety text summarises reported side effects (SIDER via DRKG) and listed interactions, not a full safety profile. &plusmn; is the spread across 3 model runs, not a confidence interval. Data: DRKG (Apache-2.0); DrugBank content is CC BY-NC 4.0. Non-commercial academic project.</div></div></div>
<div class="shade" id="sh"></div><aside class="drawer" id="dr"></aside><div class="toast" id="ts"></div>
<script>
const D=__DATA__,SAFE=__SAFE__,FEED=__FEED__,META=__META__;
const $=(s,r=document)=>r.querySelector(s),$$=(s,r=document)=>[...r.querySelectorAll(s)];
const esc=x=>String(x).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const enc=encodeURIComponent,pm=(...t)=>'https://pubmed.ncbi.nlm.nih.gov/?term='+enc(t.map(x=>'"'+x+'"').join(' AND '));
const DIS=Object.keys(D.dis).sort(),DRG=Object.keys(D.drg).sort(),fmt=n=>n.toLocaleString('en-US');
let S={v:'home',dis:'',drg:'',q:'',F:{min:-99,novel:false,noser:false,nowarn:false,top:25},cm:'dis',ca:'',cb:'',fq:'',fn:30,list:[],ctx:null,tab:0,step:0};
try{S.list=JSON.parse(localStorage.getItem('drf_list')||'[]')}catch(e){}
const save=()=>{try{localStorage.setItem('drf_list',JSON.stringify(S.list))}catch(e){}};
function toast(t){const e=$('#ts');e.textContent=t;e.classList.add('on');clearTimeout(toast.t);toast.t=setTimeout(()=>e.classList.remove('on'),1800)}
function ps(t){t=t||'';const m=t.match(/(\d+) reported side effects(?: incl\. ([^|]+))?/),d=t.match(/interacts with (\d+)/);return{n:m?+m[1]:0,fl:m&&m[2]?m[2].split(',').map(s=>s.trim()):[],warn:/WARNING/.test(t),ddi:d?+d[1]:0,none:/no side-effect data/.test(t)}}
function badges(t){const p=ps(t);let h='';if(p.fl.length)h+='<span class="chip r">&#9888; '+p.fl.length+' serious-type effect'+(p.fl.length>1?'s':'')+'</span>';if(p.warn)h+='<span class="chip a">inflation risk</span>';if(p.ddi)h+='<span class="chip">'+p.ddi+' interactions</span>';if(p.none)h+='<span class="chip">no side-effect data</span>';return h}
const inList=(d,x)=>S.list.some(i=>i.d==d&&i.x==x);
const NAV=[['home','&#9673;','Overview'],['dis','&#9877;','Find drugs'],['drg','&#9879;','Find diseases'],['cmp','&#8644;','Compare'],['feed','&#10024;','Discovery feed'],['list','&#9733;','Shortlist'],['model','&#9881;','Model &amp; method']];
function nav(){$('#nav').innerHTML='<div class="brand"><div class="logo">Rx</div><b>Drug Repurposing<br>Finder</b></div>'+NAV.map(n=>'<button class="nav'+(S.v==n[0]?' on':'')+'" data-v="'+n[0]+'"><i>'+n[1]+'</i><span>'+n[2]+'</span>'+(n[0]=='list'&&S.list.length?'<em class="ct">'+S.list.length+'</em>':'')+'</button>').join('')+'<div class="sp"></div><div class="mu sm" style="padding:8px">Private research tool<br>DRKG knowledge graph + graph neural network</div>'}
function go(v){S.v=v;closeD();nav();render();scrollTo(0,0)}
function bars(it){if(!it.length)return'';const mx=Math.max(...it.map(i=>i[1]+i[2])),mn=Math.min(0,...it.map(i=>i[1]-i[2])),W=640,L=190,B=380,x=v=>L+(v-mn)/((mx-mn)||1)*B,H=it.length*28+8;
return '<svg viewBox="0 0 '+W+' '+H+'" style="width:100%">'+it.map((i,k)=>{const y=k*28+4;return '<text x="'+(L-8)+'" y="'+(y+17)+'" text-anchor="end" font-size="12.5" fill="var(--fg)">'+esc(i[0].length>26?i[0].slice(0,25)+'…':i[0])+'</text><rect x="'+x(Math.min(0,i[1]))+'" y="'+y+'" width="'+Math.max(2,Math.abs(x(i[1])-x(0)))+'" height="20" rx="4" fill="'+(i[3]?'var(--ac)':'var(--ac2)')+'"/><line x1="'+x(i[1]-i[2])+'" x2="'+x(i[1]+i[2])+'" y1="'+(y+10)+'" y2="'+(y+10)+'" stroke="var(--fg)" stroke-width="1.6"/><text x="'+(x(i[1]+i[2])+6)+'" y="'+(y+15)+'" font-size="12" fill="var(--mu)">'+i[1]+'</text>'}).join('')+'</svg>'}
const SRC={dis:()=>S.dis,drg:()=>S.drg};
function pickBox(kind){const L=kind=='dis'?DIS:DRG,q=S.q.toLowerCase(),f=L.filter(n=>n.toLowerCase().includes(q)).slice(0,200),sel=S[kind];
return '<div class="card"><div class="eyebrow">'+(kind=='dis'?'Choose a disease':'Choose a drug')+'</div><input type="text" id="pq" value="'+esc(S.q)+'" placeholder="Type to filter ('+fmt(L.length)+')"><div class="list">'+f.map(n=>'<button data-p="'+esc(n)+'" class="'+(n==sel?'on':'')+'"><span>'+esc(n)+'</span>'+(kind=='dis'?'<span class="tag">'+D.dis[n].known+' known</span>':'')+'</button>').join('')+'</div><button class="btn" style="margin-top:10px;width:100%" id="rnd">&#127922; Surprise me</button></div>'}
function table(rows,dm,ctxf){return '<table><thead><tr><th>#</th><th>'+(dm?'Drug':'Disease')+'</th><th>Score &plusmn; spread</th><th class="hide-m">Evidence genes</th><th></th></tr></thead><tbody>'+rows.map((r,i)=>{const e=r[5];return '<tr class="rw'+(r[4]?' k':'')+'" data-i="'+i+'"><td>'+r[0]+'</td><td><b>'+esc(r[1])+'</b>'+(r[4]?' <span class="chip g">known use</span>':'')+'<div class="mu sm">'+esc(r[2])+'</div>'+(dm?badges(r[6]):'')+'</td><td>'+r[3]+' <span class="mu sm">&plusmn;'+r[7]+'</span><div class="bar" style="width:'+Math.max(5,Math.min(100,(r[3]+2)*16))+'%"></div></td><td class="hide-m sm">'+e.d.map(g=>'<span class="chip g">'+esc(g)+'</span>').join('')+e.v.slice(0,2).map(p=>'<span class="chip">'+esc(p[0])+'~'+esc(p[1])+'</span>').join('')+(e.d.length+e.v.length?'':'<span class="mu">none named</span>')+'</td><td><a href="'+(dm?pm(r[1],S.dis):pm(S.drg,r[1]))+'" target="_blank" rel="noopener" class="sm">PubMed</a> <button class="star'+(inList(ctxf(r).drug,ctxf(r).disease)?' on':'')+'" data-s="'+i+'" title="Add to shortlist">&#9733;</button></td></tr>'}).join('')+'</tbody></table>'}
function csv(rows,dm,name){const e=x=>'"'+String(x).replace(/"/g,'""')+'"',c=[['rank',dm?'drug':'disease','id','score','seed_spread','known_use','direct_genes'].join(',')].concat(rows.map(r=>[r[0],e(r[1]),e(r[2]),r[3],r[7],r[4],e(r[5].d.join('; '))].join(','))).join('\n'),a=document.createElement('a');a.href=URL.createObjectURL(new Blob([c],{type:'text/csv'}));a.download=name.replace(/[^A-Za-z0-9]+/g,'_')+'.csv';a.click()}
let CUR=[];
function vFind(kind){const dm=kind=='dis',sel=S[kind],d=sel&&D[kind][sel];let h='<div class="eyebrow">'+(dm?'Disease → ranked drugs':'Drug → ranked diseases')+'</div><h2 style="margin:4px 0 14px;font-size:28px">'+(dm?'Which existing drugs may help?':'What else might this drug treat?')+'</h2><div class="pick">'+pickBox(kind)+'<div>';
if(!d)return h+'<div class="card mu">Pick an item on the left, or press Surprise me.</div></div></div>';
let rows=d.rows.filter(r=>r[3]>=S.F.min&&(!S.F.novel||!r[4])&&(!dm||!S.F.noser||!ps(r[6]).fl.length)&&(!dm||!S.F.nowarn||!ps(r[6]).warn)).slice(0,S.F.top);CUR=rows;
h+='<div class="card"><div style="display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap"><div><h3 style="font-size:24px">'+esc(sel)+'</h3><div class="mu sm">'+(dm?'MeSH '+esc(d.id)+' · '+d.known+' drugs with a known DrugBank use':'DrugBank '+esc(d.id))+'</div></div><div><button class="btn" id="dl">&#8681; CSV</button> <a class="btn" style="text-decoration:none;display:inline-block" target="_blank" rel="noopener" href="'+pm(sel)+'">PubMed</a></div></div>'+(dm?'':'<div style="margin-top:8px">'+badges(SAFE[sel])+'</div>')+'<div class="flt"><label>Show top <input type="range" id="ft" min="5" max="'+(dm?25:12)+'" value="'+Math.min(S.F.top,dm?25:12)+'"><b>'+Math.min(S.F.top,rows.length)+'</b></label><label>Min score <input type="range" id="fm" min="-2" max="4" step=".25" value="'+Math.max(S.F.min,-2)+'"><b>'+(S.F.min<-2?'any':S.F.min)+'</b></label><label><input type="checkbox" id="fn"'+(S.F.novel?' checked':'')+'> novel only (not in DrugBank)</label>'+(dm?'<label><input type="checkbox" id="fs"'+(S.F.noser?' checked':'')+'> hide serious-effect flags</label><label><input type="checkbox" id="fw"'+(S.F.nowarn?' checked':'')+'> hide inflation-risk</label>':'')+'</div>';
h+=rows.length?'<div class="mu sm" style="margin-bottom:6px">Bar colour: <span style="color:var(--ac)">&#9632;</span> already known in DrugBank, <span style="color:var(--ac2)">&#9632;</span> novel suggestion. Whiskers show &plusmn; spread across 3 runs.</div>'+bars(rows.slice(0,10).map(r=>[r[1],r[3],r[7],r[4]]))+table(rows,dm,r=>dm?{drug:r[1],disease:sel}:{drug:sel,disease:r[1]})+'<div class="mu sm" style="margin-top:8px">Click any row for evidence graph, score breakdown, safety and literature.</div>':'<p class="mu">No rows match these filters.</p>';
return h+'</div></div></div>'}
function ctxOf(r,kind){return kind=='dis'?{drug:r[1],disease:S.dis,row:r}:{drug:S.drg,disease:r[1],row:r}}
function vCmp(){const dm=S.cm=='dis',L=dm?DIS:DRG,A=D[dm?'dis':'drg'][S.ca],B=D[dm?'dis':'drg'][S.cb];
let h='<div class="eyebrow">Compare</div><h2 style="margin:4px 0 14px;font-size:28px">Side-by-side comparison</h2><div class="card"><div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px"><button class="btn'+(dm?' pri':'')+'" data-cm="dis">Two diseases</button><button class="btn'+(!dm?' pri':'')+'" data-cm="drg">Two drugs</button></div><div class="grid g2"><div><input type="text" list="cl" id="ca" value="'+esc(S.ca)+'" placeholder="First '+(dm?'disease':'drug')+'…"></div><div><input type="text" list="cl" id="cb" value="'+esc(S.cb)+'" placeholder="Second '+(dm?'disease':'drug')+'…"></div></div><datalist id="cl">'+L.map(n=>'<option value="'+esc(n)+'">').join('')+'</datalist></div>';
if(!A||!B)return h+'<div class="card mu" style="margin-top:14px">Choose two items to see which '+(dm?'drugs':'diseases')+' they share.</div>';
const ma=new Map(A.rows.map(r=>[r[1],r])),mb=new Map(B.rows.map(r=>[r[1],r])),sh=[...ma.keys()].filter(k=>mb.has(k)),oa=[...ma.keys()].filter(k=>!mb.has(k)),ob=[...mb.keys()].filter(k=>!ma.has(k)),jac=Math.round(100*sh.length/(ma.size+mb.size-sh.length||1));
const ch=(k,m,o)=>'<span class="chip btn2 '+(m.get(k)[4]?'g':'')+'" data-cp="'+esc(k)+'" data-side="'+o+'">'+esc(k)+' <b>'+m.get(k)[3]+'</b></span>';
h+='<div class="grid g3" style="margin-top:14px"><div class="card"><div class="eyebrow">Only in '+esc(S.ca)+'</div><div style="margin-top:8px">'+(oa.map(k=>ch(k,ma,'a')).join('')||'<span class="mu">-</span>')+'</div></div><div class="card" style="border-color:var(--ac2)"><div class="eyebrow">Shared · overlap '+jac+'%</div><div style="margin-top:8px">'+(sh.length?sh.map(k=>'<span class="chip btn2 g" data-cp="'+esc(k)+'" data-side="a">'+esc(k)+' <b>'+ma.get(k)[3]+' / '+mb.get(k)[3]+'</b></span>').join(''):'<span class="mu">nothing shared in the top lists</span>')+'</div><div class="mu sm" style="margin-top:8px">A shared '+(dm?'drug':'disease')+' hints that the two '+(dm?'diseases':'drugs')+' share biology.</div></div><div class="card"><div class="eyebrow">Only in '+esc(S.cb)+'</div><div style="margin-top:8px">'+(ob.map(k=>ch(k,mb,'b')).join('')||'<span class="mu">-</span>')+'</div></div></div><div class="mu sm" style="margin-top:8px">Chips show the model score (shared: score in A / score in B). Click a chip for details.</div>';return h}
function vFeed(){const q=S.fq.toLowerCase(),f=FEED.filter(r=>!q||(r[1]+' '+r[10]).toLowerCase().includes(q)),sh=f.slice(0,S.fn);
return '<div class="eyebrow">Discovery feed</div><h2 style="margin:4px 0 6px;font-size:28px">Top novel suggestions across everything</h2><p class="mu">The highest-scored drug–disease pairs that are <b>not</b> already listed in DrugBank. These are the leads most worth reading about.</p><div class="card" style="margin-top:12px"><input type="text" id="fq" value="'+esc(S.fq)+'" placeholder="Filter by drug or disease…"><div class="mu sm" style="margin:8px 0">'+f.length+' of '+FEED.length+' leads</div><div class="grid g2">'+sh.map(r=>'<button class="card" style="text-align:left;padding:14px" data-f="'+FEED.indexOf(r)+'"><div class="eyebrow">#'+r[0]+' · score '+r[3]+' &plusmn;'+r[7]+'</div><div style="font-size:17px;margin:4px 0"><b>'+esc(r[1])+'</b> &rarr; '+esc(r[10])+'</div><div>'+r[5].d.slice(0,3).map(g=>'<span class="chip g">'+esc(g)+'</span>').join('')+badges(r[6])+'</div></button>').join('')+'</div>'+(f.length>S.fn?'<button class="btn" id="more" style="margin-top:12px">Show more</button>':'')+'</div>'}
function vList(){if(!S.list.length)return '<div class="eyebrow">Shortlist</div><h2 style="margin:4px 0 14px;font-size:28px">Your shortlist</h2><div class="card mu">Nothing saved yet. Click the &#9733; next to any suggestion to collect it here.</div>';
return '<div class="eyebrow">Shortlist</div><h2 style="margin:4px 0 14px;font-size:28px">Your shortlist ('+S.list.length+')</h2><div class="card"><div style="display:flex;gap:8px;margin-bottom:10px"><button class="btn" id="lx">&#8681; Export CSV</button><button class="btn" id="lc">Clear all</button></div><table><thead><tr><th>Drug</th><th>Disease</th><th>Score</th><th></th></tr></thead><tbody>'+S.list.map((i,k)=>'<tr><td><b>'+esc(i.d)+'</b></td><td>'+esc(i.x)+'</td><td>'+i.s+'</td><td><a class="sm" target="_blank" rel="noopener" href="'+pm(i.d,i.x)+'">PubMed</a> <button class="star" data-rm="'+k+'">&#10005;</button></td></tr>').join('')+'</tbody></table></div>'}
const STEPS=[['Load','pandas','Read DRKG, attach names (Hetionet, MeSH, SIDER). Delete every direct drug-disease edge and all ATC / drug-class data.','A clean graph with no answers and no shortcuts.'],['Build','SciPy · NumPy','Turn the graph into sparse grids: drug-gene, gene-disease, gene-gene. Make 5 cold-drug folds.','The graph as numbers; every drug is tested once, unseen.'],['Baseline','scikit-learn','Gradient boosting on path counts (drug→gene→disease) and look-alike-drug evidence.','The score to beat: MRR 0.203.'],['Learn','PyTorch · GNN','A graph neural network builds each drug from the genes it acts on, using gene-structure (SVD) features and a label-free drug descriptor.','MRR 0.227 alone; works for drugs never seen.'],['Blend','NumPy','Mix the two models (z-scores) and average 3 training seeds.','MRR 0.253 (blend), 0.269 (3 seeds).'],['Rank','json · HTML','Score all '+fmt(META.pairs)+' drug-disease pairs and attach evidence genes, side effects, interactions and PubMed links.','This dashboard.']];
function vModel(){const M=[['Boosting baseline',.203],['Graph network',.227],['Blend',.253],['Blend × 3 seeds',.269]],st=STEPS[S.step];
return '<div class="eyebrow">Model &amp; method</div><h2 style="margin:4px 0 14px;font-size:28px">How the model was built and tested</h2><div class="card"><div class="eyebrow">Pipeline: click a step</div><div class="pipe" style="margin-top:10px">'+STEPS.map((s,i)=>'<button data-st="'+i+'" class="'+(i==S.step?'on':'')+'"><b>'+(i+1)+'</b>'+s[0]+'<div class="sm" style="opacity:.8">'+s[1]+'</div></button>').join('')+'</div><div style="margin-top:14px" class="grid g2"><div><b>What happens</b><p class="mu">'+st[2]+'</p></div><div><b>Outcome</b><p class="mu">'+st[3]+'</p></div></div></div>'+
'<div class="grid g2" style="margin-top:14px"><div class="card"><div class="eyebrow">Accuracy on drugs never seen in training</div><p class="mu sm">MRR = average of 1 / rank of the true disease. Higher is better; chance is about 0.009.</p><svg viewBox="0 0 400 190" style="width:100%">'+M.map((m,i)=>'<rect x="'+(20+i*95)+'" y="'+(160-m[1]*520)+'" width="70" height="'+m[1]*520+'" rx="6" fill="'+(i==3?'var(--ac2)':'var(--ac)')+'" opacity="'+(.45+i*.18)+'"/><text x="'+(55+i*95)+'" y="'+(152-m[1]*520)+'" text-anchor="middle" font-size="14" font-weight="700" fill="var(--fg)">'+m[1]+'</text><text x="'+(55+i*95)+'" y="178" text-anchor="middle" font-size="10.5" fill="var(--mu)">'+m[0]+'</text>').join('')+'</svg></div><div class="card"><div class="eyebrow">Final model (3 seeds)</div><div class="grid g3" style="margin-top:10px"><div class="stat"><b>45.8%</b><span>true disease in top 10 of 762</span></div><div class="stat"><b>66.3%</b><span>in top 50</span></div><div class="stat"><b>14</b><span>median rank</span></div></div><p class="mu sm" style="margin-top:10px">Single blend vs baseline was confirmed with bootstrap 95% confidence intervals; the 3-seed average is an observed further gain.</p></div></div>'+
'<div class="grid g3" style="margin-top:14px"><div class="card"><div class="eyebrow">Data</div><div class="stat"><b>'+fmt(META.nc)+'</b><span>drugs</span></div><div class="stat"><b>'+fmt(META.ng)+'</b><span>genes and proteins</span></div><div class="stat"><b>'+META.nd+'</b><span>diseases</span></div><div class="stat"><b>'+fmt(META.labels)+'</b><span>known pairs (answer key, '+fmt(META.labdrugs)+' drugs)</span></div></div><div class="card"><div class="eyebrow">Honest testing</div><details open style="margin-top:8px"><summary>Cold-drug split</summary><p class="mu sm">Test drugs are hidden together with all their known uses, in 5 rounds.</p></details><details style="margin-top:6px"><summary>No answers</summary><p class="mu sm">Every direct drug-disease link was removed.</p></details><details style="margin-top:6px"><summary>No hints</summary><p class="mu sm">ATC / drug-class codes removed: a drug\'s class already tells what it treats.</p></details></div><div class="card"><div class="eyebrow">Limits</div><details open style="margin-top:8px"><summary>Incomplete labels</summary><p class="mu sm">A suggestion marked unknown may be real.</p></details><details style="margin-top:6px"><summary>Over-ranked compounds</summary><p class="mu sm">Simple compounds with many links can rank high (see inflation-risk flag).</p></details><details style="margin-top:6px"><summary>No clinical testing</summary><p class="mu sm">Everything here is a research lead.</p></details></div></div>'}
function vHome(){const pop=[...DIS].sort((a,b)=>D.dis[b].known-D.dis[a].known).slice(0,8),st=[[META.pairs,'drug–disease pairs scored'],[META.pool,'drugs searchable'],[META.nd,'diseases'],[META.ng,'genes in the graph']];
return '<div class="hero"><div style="position:relative;z-index:1"><div class="eyebrow" style="color:#f28b6a">AI-based drug discovery &amp; repurposing</div><h1>Which existing medicine could treat which disease?</h1><p>A knowledge graph of drugs, genes and diseases, a graph neural network, and an evidence trail for every suggestion. Built to be tested honestly on drugs it has never seen.</p><div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:14px"><button class="btn pri" data-v="dis">Find drugs for a disease</button><button class="btn" style="color:#14262b" data-v="drg">Find diseases for a drug</button><button class="btn" style="color:#14262b" id="sur">&#127922; Surprise me</button></div></div><svg viewBox="0 0 420 300" style="width:100%;position:relative;z-index:1"><g stroke="#7fd1c4" stroke-opacity=".5" class="fl" fill="none"><path d="M50 60L210 40M50 60L210 130M50 150L210 130M50 150L210 220M50 240L210 220M50 240L210 270M210 40L370 90M210 130L370 90M210 130L370 180M210 220L370 180M210 270L370 230"/></g><path d="M50 150L210 130L370 180" stroke="#f28b6a" stroke-width="4" fill="none"/><g fill="#f28b6a"><circle cx="50" cy="60" r="12"/><circle cx="50" cy="150" r="16" stroke="#fff" stroke-width="3"/><circle cx="50" cy="240" r="12"/></g><g fill="#7fd1c4" class="pu"><circle cx="210" cy="40" r="9"/><circle cx="210" cy="130" r="12" stroke="#fff" stroke-width="3"/><circle cx="210" cy="220" r="9"/><circle cx="210" cy="270" r="9"/></g><g fill="#f5f2ea"><circle cx="370" cy="90" r="14"/><circle cx="370" cy="180" r="17"/><circle cx="370" cy="230" r="13"/></g><text x="50" y="190" fill="#f5f2ea" font-size="12" text-anchor="middle">drug</text><text x="210" y="168" fill="#f5f2ea" font-size="12" text-anchor="middle">gene</text><text x="370" y="216" fill="#f5f2ea" font-size="12" text-anchor="middle">disease</text></svg></div>'+
'<div class="grid g4" style="margin-top:14px">'+st.map(s=>'<div class="card stat"><b data-n="'+s[0]+'">0</b><span>'+s[1]+'</span></div>').join('')+'</div><div class="grid g2" style="margin-top:14px"><div class="card"><div class="eyebrow">Popular diseases (most known drugs)</div><div style="margin-top:8px">'+pop.map(n=>'<span class="chip btn2" data-d="'+esc(n)+'">'+esc(n)+'</span>').join('')+'</div></div><div class="card"><div class="eyebrow">Take the tour</div><div style="margin-top:8px"><span class="chip btn2" data-v="dis">1 · Pick a disease</span><span class="chip btn2" data-v="cmp">2 · Compare</span><span class="chip btn2" data-v="feed">3 · Discovery feed</span><span class="chip btn2" data-v="list">4 · Shortlist</span><span class="chip btn2" data-v="model">5 · How it was built</span></div></div></div>'+
'<div class="card" style="margin-top:14px"><div class="eyebrow">What every suggestion comes with</div><div class="grid g4" style="margin-top:10px"><div><b>Score + spread</b><div class="mu sm">relative plausibility and run-to-run agreement</div></div><div><b>Evidence graph</b><div class="mu sm">the genes linking drug and disease</div></div><div><b>Safety flags</b><div class="mu sm">side effects, interactions, inflation risk</div></div><div><b>Literature link</b><div class="mu sm">one click to PubMed</div></div></div></div>'}
function render(){const v=$('#v');v.innerHTML=({home:vHome,dis:()=>vFind('dis'),drg:()=>vFind('drg'),cmp:vCmp,feed:vFeed,list:vList,model:vModel})[S.v]();$$('[data-n]').forEach(e=>{const t=+e.dataset.n,t0=performance.now(),f=n=>{const p=Math.min(1,(n-t0)/900);e.textContent=fmt(Math.round(t*(1-Math.pow(1-p,3))));if(p<1)requestAnimationFrame(f)};requestAnimationFrame(f)})}
function node(x,y,l,c,w){return '<g class="nd '+c+'"><title>'+esc(l)+'</title><rect x="'+(x-w/2)+'" y="'+(y-14)+'" width="'+w+'" height="28" rx="14"/><text x="'+x+'" y="'+(y+4.5)+'" text-anchor="middle">'+esc(l.length>15?l.slice(0,14)+'…':l)+'</text></g>'}
function evg(c){const e=c.row[5],it=[...e.d.map(g=>({g})),...e.v.map(p=>({p}))];if(!it.length)return'<p class="mu">No named gene evidence for this pair. The score then comes mostly from look-alike drugs and unnamed genes.</p>';
const W=560,H=Math.max(170,it.length*52+40),cy=H/2,ys=it.map((_,i)=>it.length==1?cy:34+i*(H-68)/(it.length-1));let s='<svg class="evg" viewBox="0 0 '+W+' '+H+'" style="width:100%">',n='';
it.forEach((o,i)=>{const y=ys[i];if(o.g){s+='<line x1="112" y1="'+cy+'" x2="250" y2="'+y+'"/><line x1="310" y1="'+y+'" x2="448" y2="'+cy+'"/>';n+=node(280,y,o.g,'gn',60)}else{s+='<line x1="112" y1="'+cy+'" x2="180" y2="'+y+'"/><line class="dsh" x1="240" y1="'+y+'" x2="320" y2="'+y+'"/><line x1="380" y1="'+y+'" x2="448" y2="'+cy+'"/>';n+=node(210,y,o.p[0],'gn',60)+node(350,y,o.p[1],'gn',60)}});
return s+n+node(60,cy,c.drug,'drug',104)+node(500,cy,c.disease,'dz',104)+'</svg><p class="mu sm">Solid path: the drug acts on a gene that is linked to the disease. Dashed: the drug\'s gene interacts with a gene linked to the disease.</p>'}
function why(c){const r=c.row,w=z=>Math.max(4,Math.min(100,(z+2)*14)),ag=r[8]>1&&r[9]>1?'Both models rank this pair highly. This is the most reliable kind of lead.':(r[8]>1||r[9]>1)?'Only one model is strongly convinced. Treat with extra care.':'Neither model is strongly convinced.';
return '<div class="grid" style="gap:12px"><div><div class="sm"><b>Final blended score</b> '+r[3]+' <span class="mu">&plusmn;'+r[7]+' across 3 runs</span></div><div class="bt"><i style="width:'+w(r[3])+'%;background:var(--ac2)"></i></div></div><div><div class="sm"><b>Boosting model</b> (path counts, look-alike drugs): '+r[8]+'</div><div class="bt"><i style="width:'+w(r[8])+'%;background:var(--ac)"></i></div></div><div><div class="sm"><b>Graph network</b> (drug built from its genes): '+r[9]+'</div><div class="bt"><i style="width:'+w(r[9])+'%;background:var(--ac)"></i></div></div></div><p style="margin-top:14px">'+ag+'</p><p class="mu sm">Numbers are z-scores: 0 is average across all pairs, +2 is two standard deviations above. '+(r[4]?'This use is already listed in DrugBank, which shows the model can recover real knowledge.':'This use is not listed in DrugBank: it is either a new opportunity or a false positive.')+'</p>'}
function saf(c){const t=SAFE[c.drug]||c.row[6]||'',p=ps(t);return '<div class="grid" style="gap:10px"><div class="card" style="padding:12px"><b>'+(p.none?'No side-effect data':p.n+' reported side effects')+'</b>'+(p.fl.length?'<div style="margin-top:6px">'+p.fl.map(f=>'<span class="chip r">&#9888; '+esc(f)+'</span>').join('')+'</div>':'')+'</div><div class="card" style="padding:12px"><b>'+p.ddi+' listed drug interactions</b></div>'+(p.warn?'<div class="card" style="padding:12px;border-color:var(--am)"><b>Inflation risk</b><div class="mu sm">This compound has unusually many gene links, so its score may be too high.</div></div>':'')+'</div><p class="mu sm" style="margin-top:12px">Source: SIDER side effects and interaction records inside DRKG. Not a complete safety profile; clinicians decide.</p>'}
function lit(c){const e=c.row[5],gs=[...new Set([...e.d,...e.v.flat()])];return '<p><a target="_blank" rel="noopener" href="'+pm(c.drug,c.disease)+'"><b>'+esc(c.drug)+' + '+esc(c.disease)+'</b></a></p><p class="mu sm">Many papers: studied, check results are positive. A few: early evidence, the most interesting case. Zero: unexplored or wrong; try simpler spellings first.</p>'+(gs.length?'<div class="eyebrow" style="margin-top:12px">Gene links</div>'+gs.slice(0,8).map(g=>'<div class="sm" style="margin-top:6px">'+esc(g)+': <a target="_blank" rel="noopener" href="'+pm(c.drug,g)+'">drug</a> · <a target="_blank" rel="noopener" href="'+pm(g,c.disease)+'">disease</a></div>').join(''):'')}
function drawer(){const c=S.ctx,d=$('#dr');if(!c){d.classList.remove('open');$('#sh').classList.remove('on');return}
const T=['Evidence','Why this score','Safety','Literature'],B=[evg,why,saf,lit];d.innerHTML='<div style="display:flex;justify-content:space-between"><div class="eyebrow">Suggestion #'+c.row[0]+(c.row[4]?' · known use':' · novel')+'</div><button class="star" id="cx" style="font-size:22px">&#10005;</button></div><h3 style="font-size:24px;margin-top:6px">'+esc(c.drug)+' <span class="mu">&rarr;</span> '+esc(c.disease)+'</h3><div style="margin-top:8px"><button class="btn" id="sv">'+(inList(c.drug,c.disease)?'&#9733; In shortlist':'&#9734; Add to shortlist')+'</button></div><div class="tabs">'+T.map((t,i)=>'<button data-t="'+i+'" class="'+(S.tab==i?'on':'')+'">'+t+'</button>').join('')+'</div>'+B[S.tab](c);d.classList.add('open');$('#sh').classList.add('on')}
function openD(c){S.ctx=c;S.tab=0;drawer()}function closeD(){S.ctx=null;drawer()}
function pickItem(kind,n){S.v=kind;S[kind]=n;S.q='';nav();render()}
document.addEventListener('click',e=>{const t=e.target,g=(s)=>t.closest(s);let b;
if(b=g('[data-v]'))return go(b.dataset.v);if(b=g('[data-d]'))return pickItem('dis',b.dataset.d);
if(b=g('[data-p]'))return pickItem(S.v,b.dataset.p);if(g('#rnd')||g('#sur')){const r=FEED[Math.floor(Math.random()*FEED.length)];S.v='dis';S.dis=r[10];nav();render();return openD({drug:r[1],disease:r[10],row:r})}
if(b=g('[data-s]')){e.stopPropagation();const r=CUR[+b.dataset.s],c=ctxOf(r,S.v);const k=S.list.findIndex(i=>i.d==c.drug&&i.x==c.disease);if(k<0){S.list.push({d:c.drug,x:c.disease,s:r[3]});toast('Added to shortlist')}else S.list.splice(k,1);save();nav();return render()}
if(g('a'))return;if(b=g('tr.rw'))return openD(ctxOf(CUR[+b.dataset.i],S.v));if(b=g('[data-f]')){const r=FEED[+b.dataset.f];return openD({drug:r[1],disease:r[10],row:r})}
if(b=g('[data-cm]')){S.cm=b.dataset.cm;S.ca=S.cb='';return render()}
if(b=g('[data-cp]')){const dm=S.cm=='dis',A=D[dm?'dis':'drg'],src=b.dataset.side=='b'?S.cb:S.ca,r=A[src].rows.find(x=>x[1]==b.dataset.cp);return openD(dm?{drug:r[1],disease:src,row:r}:{drug:src,disease:r[1],row:r})}
if(b=g('[data-st]')){S.step=+b.dataset.st;return render()}if(b=g('[data-t]')){S.tab=+b.dataset.t;return drawer()}
if(g('#cx')||g('#sh'))return closeD();if(g('#dl'))return csv(CUR,S.v=='dis',S[S.v]);if(g('#more')){S.fn+=30;return render()}
if(g('#sv')){const c=S.ctx,k=S.list.findIndex(i=>i.d==c.drug&&i.x==c.disease);if(k<0){S.list.push({d:c.drug,x:c.disease,s:c.row[3]});toast('Added to shortlist')}else S.list.splice(k,1);save();nav();drawer();if(S.v=='dis'||S.v=='drg')render();return}
if(b=g('[data-rm]')){S.list.splice(+b.dataset.rm,1);save();nav();return render()}if(g('#lc')){S.list=[];save();nav();return render()}
if(g('#lx')){const c='drug,disease,score\n'+S.list.map(i=>'"'+i.d+'","'+i.x+'",'+i.s).join('\n'),a=document.createElement('a');a.href=URL.createObjectURL(new Blob([c],{type:'text/csv'}));a.download='shortlist.csv';a.click();return}
if(!g('.gs'))$('#sug').hidden=true});
document.addEventListener('input',e=>{const t=e.target;
if(t.id=='pq'){S.q=t.value;const p=t.selectionStart;render();const n=$('#pq');n.focus();n.setSelectionRange(p,p)}
if(t.id=='ft'){S.F.top=+t.value;rr()}if(t.id=='fm'){S.F.min=+t.value;rr()}
if(t.id=='fq'){S.fq=t.value;S.fn=30;const p=t.selectionStart;render();const n=$('#fq');n.focus();n.setSelectionRange(p,p)}
if(t.id=='ca'||t.id=='cb'){const k=t.id=='ca'?'ca':'cb';S[k]=t.value;const L=S.cm=='dis'?D.dis:D.drg;if(L[t.value]){render()}}
if(t.id=='gs'){const q=t.value.toLowerCase().trim(),u=$('#sug');if(!q){u.hidden=true;return}const m=[...DIS.filter(n=>n.toLowerCase().includes(q)).slice(0,5).map(n=>['dis',n]),...DRG.filter(n=>n.toLowerCase().includes(q)).slice(0,5).map(n=>['drg',n])].slice(0,8);u.innerHTML=m.map(x=>'<button data-g="'+x[0]+'" data-gn="'+esc(x[1])+'"><span class="tag">'+(x[0]=='dis'?'disease':'drug')+'</span>'+esc(x[1])+'</button>').join('')||'<div class="mu" style="padding:10px 14px">No match</div>';u.hidden=false}});
document.addEventListener('change',e=>{const t=e.target;if(t.id=='fn'){S.F.novel=t.checked;rr()}if(t.id=='fs'){S.F.noser=t.checked;rr()}if(t.id=='fw'){S.F.nowarn=t.checked;rr()}});
function rr(){const y=scrollY;render();scrollTo(0,y)}
$('#sug').addEventListener('click',e=>{const b=e.target.closest('[data-g]');if(b){$('#sug').hidden=true;$('#gs').value='';pickItem(b.dataset.g,b.dataset.gn)}});
document.addEventListener('keydown',e=>{if(e.key=='/'&&!/INPUT|TEXTAREA/.test(document.activeElement.tagName)){e.preventDefault();$('#gs').focus()}if(e.key=='Escape'){closeD();$('#sug').hidden=true}});
$('#th').onclick=()=>{const r=document.documentElement,d=getComputedStyle(r).getPropertyValue('--bg').trim()=='#0b1c21';r.dataset.theme=d?'light':'dark'};
nav();render();
</script></body></html>"""
out = WORK / "dashboard.html"
J = lambda o: json.dumps(o).replace("</", "<\\/")
out.write_text(page.replace("__DATA__", J(dict(dis=dis, drg=drg))).replace("__SAFE__", J(SAFE)).replace("__FEED__", J(FEED)).replace("__META__", J(META)), encoding="utf-8")
print("PRO dashboard saved:", out, f"({out.stat().st_size/1e6:.1f} MB)")